# Transformer 预处理

欢迎来到第 4 周的第一个不计分实验室。在本 Notebook 中，你将深入研究在将原始文本传递到 Transformer 架构的编码器和解码器块之前应用于原始文本的预处理方法。 

**在此作业之后，你将能够**：

* 创建可视化以获得对位置编码的直觉
* 可视化位置编码如何影响词嵌入


## 目录

- [依赖库](#0)
- [1 - 位置编码](#1)
    - [1.1 - 位置编码可视化](#1-1)
    - [1.2 - 比较位置编码](#1-2)   
- [2 - 语义嵌入](#2)
    - [2.1 - 加载预训练嵌入](#2-1)
    - [2.2 - 笛卡尔平面上的可视化](#2-2)
- [3 - 语义和位置嵌入](#3)

<a name='0'></a>
## 依赖库

运行以下单元格来加载你需要的包。

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os

from tensorflow.keras.layers import Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

<a name='1'></a>
## 1 - 位置编码

下面给出你在上一份作业中实现的位置编码公式。该编码使用以下公式：

$$
PE_{(pos, 2i)}= sin\left(\frac{pos}{{10000}^{\frac{2i}{d}}}\right)
$$

$$
PE_{(pos, 2i+1)}= cos\left(\frac{pos}{{10000}^{\frac{2i}{d}}}\right)
$$

<a name='ex-2'></a>


在将文本输入语言模型之前，将句子转换为标记是自然语言处理任务的标准做法。然后，每个标记都被转换为固定长度的数值向量（称为嵌入），它捕获单词的含义。在 Transformer 架构中，位置编码向量被添加到嵌入中，以在整个模型中传递位置信息。 

仅通过检查数字表示很难掌握这些向量的含义，但可视化可以帮助人们直观地了解单词的语义和位置相似性。正如你在之前的作业中所看到的，当嵌入被简化为二维并绘制时，语义相似的单词看起来靠得更近，而不同的单词则被绘制得更远。可以使用位置编码向量执行类似的练习 - 句子中较近的单词在绘制在笛卡尔平面上时应显得更近，而在句子中较远时应在平面上显得较远。 

在此 Notebook 中，你将创建词嵌入和位置编码向量的一系列可视化，以直观地了解位置编码如何影响词嵌入，并帮助通过 Transformer 架构传输顺序信息。  
    



<a name='1-1'></a>
### 1.1 - 位置编码可视化

以下代码单元具有你在 Transformer 作业中实现的 `positional_encoding` 函数。干得好！你将在此基础上使用此 Notebook 中的函数创建更多可视化效果。

In [ ]:
def positional_encoding(positions, d):
    """
    使用所有位置编码预先计算矩阵 
    
    参数：
        Positions (int) -- 要编码的最大位置数 
        d (int) -- 编码大小 
    
    返回：
        pos_encoding -- (1, position, d_model) 具有位置编码的矩阵
    """

    # 初始化所有角度的矩阵 angle_rads 
    angle_rads = np.arange(positions)[:, np.newaxis] / np.power(10000, (2 * (np.arange(d)[np.newaxis, :]//2)) / np.float32(d))
    angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])
    angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])
    
    pos_encoding = angle_rads[np.newaxis, ...]
    
    return tf.cast(pos_encoding, dtype=tf.float32)

将嵌入维度定义为100。该值必须与词嵌入的维度匹配。在 ["Attention is All You Need"](https://arxiv.org/abs/1706.03762) 论文中，嵌入大小范围从 100 到 1024，具体取决于任务。作者还根据任务使用 40 到 512 之间的最大序列长度。定义最大序列长度为 100，最大单词数为 64。 

In [ ]:
EMBEDDING_DIM = 100
MAX_SEQUENCE_LENGTH = 100
MAX_NB_WORDS = 64
pos_encoding = positional_encoding(MAX_SEQUENCE_LENGTH, EMBEDDING_DIM)

plt.pcolormesh(pos_encoding[0], cmap='RdBu')
plt.xlabel('d')
plt.xlim((0, EMBEDDING_DIM))
plt.ylabel('Position')
plt.colorbar()
plt.show()

你已经在此作业中创建了此可视化，但让我们更深入地研究一下。请注意矩阵的一些有趣的属性 - 第一个是每个向量的范数始终是常数。无论 `pos` 的值是多少，范数始终是相同的值，在本例中为 7.071068。从这个属性可以得出结论，两个位置编码向量的点积不受向量尺度的影响，这对于相关性计算具有重要意义。 

In [ ]:
pos = 34
tf.norm(pos_encoding[0,pos,:])

另一个有趣的属性是由 `k` 位置分隔的 2 个向量之间的差的范数也是恒定的。如果保持 `k` 不变并更改 `pos`，则差异将具有大致相同的值。这个属性很重要，因为它表明差异并不取决于每个编码的位置，而是取决于编码之间的相对分离。能够将位置编码表示为彼此的线性函数可以帮助模型通过关注单词的相对位置来学习。

用向量编码来反映单词位置的差异是很难实现的，特别是考虑到向量编码的值必须保持足够小，以便它们不会扭曲词嵌入。 

In [ ]:
pos = 70
k = 2
print(tf.norm(pos_encoding[0,pos,:] -  pos_encoding[0,pos + k,:]))

你已经观察到了有关位置编码向量的一些有趣的属性 - 接下来，你将创建一些可视化来查看这些属性如何影响编码和嵌入之间的关系！

<a name='1-2'></a>
### 1.2 - 比较位置编码

#### 1.2.1 - 相关性

位置编码矩阵有助于可视化每个向量对于每个位置的唯一性。然而，目前还不清楚这些向量如何表示句子中单词的相对位置。为了说明这一点，你将计算每个位置处的向量对之间的相关性。成功的位置编码器将产生一个完全对称的矩阵，其中最大值位于主对角线上 - 相似位置的向量应具有最高的相关性。遵循相同的逻辑，相关值随着远离主对角线而变小。


In [ ]:
# 位置编码相关性
corr = tf.matmul(pos_encoding, pos_encoding, transpose_b=True).numpy()[0]
plt.pcolormesh(corr, cmap='RdBu')
plt.xlabel('Position')
plt.xlim((0, MAX_SEQUENCE_LENGTH))
plt.ylabel('Position')
plt.colorbar()
plt.show()

#### 1.2.2 - 欧几里德距离

你还可以使用欧氏距离代替相关性来比较位置编码向量。在这种情况下，你的可视化将显示一个矩阵，其中主对角线为 0，其非对角线值随着远离主对角线而增加。


In [ ]:
# 位置编码欧氏距离
eu = np.zeros((MAX_SEQUENCE_LENGTH, MAX_SEQUENCE_LENGTH))
print(eu.shape)
for a in range(MAX_SEQUENCE_LENGTH):
    for b in range(a + 1, MAX_SEQUENCE_LENGTH):
        eu[a, b] = tf.norm(tf.math.subtract(pos_encoding[0, a], pos_encoding[0, b]))
        eu[b, a] = eu[a, b]
        
plt.pcolormesh(eu, cmap='RdBu')
plt.xlabel('Position')
plt.xlim((0, MAX_SEQUENCE_LENGTH))
plt.ylabel('Position')
plt.colorbar()
plt.show()

干得好！你可以使用这些可视化来检查你创建的任何位置编码。 

<a name='2'></a>
## 2 - 语义嵌入

通过创建相关矩阵和距离矩阵，你已经深入了解了位置编码向量与不同位置的其他向量之间的关系。同样，通过可视化这些向量的总和，你可以更直观地了解位置编码如何影响词嵌入。  


<a name='2-1'></a>
### 2.1 - 加载预训练嵌入

要将预训练词嵌入与你创建的位置编码结合起来，首先从 [手套](https://nlp.stanford.edu/projects/glove/) 项目加载预训练嵌入之一。你将使用具有 100 个特征的嵌入。 

In [ ]:
embeddings_index = {}
GLOVE_DIR = "glove"
f = open(os.path.join(GLOVE_DIR, 'glove.6B.100d.txt'))
for line in f:
    values = line.split()
    word = values[0]
    coefs = np.asarray(values[1:], dtype='float32')
    embeddings_index[word] = coefs
f.close()

print('Found %s word vectors.' % len(embeddings_index))
print('d_model:', embeddings_index['hi'].shape)

**注意：** 该嵌入由 400,000 个单词组成，每个词嵌入有 100 个特征。

考虑以下仅包含两个句子的文本。等一下——这些句子没有任何意义！相反，这些句子的设计是这样的：
* 每个句子都由单词组组成，每个组之间具有一些语义相似性。
* 在第一个句子中，相似的术语是连续的，而在第二个句子中，顺序是随机的。 

In [ ]:
texts = ['king queen man woman dog wolf football basketball red green yellow',
         'man queen yellow basketball green dog  woman football  king red wolf']

首先，运行以下代码单元以将分词应用于原始文本。不要太担心这一步的作用 - 它将在后面的不计分实验中详细解释。快速总结（对于理解实验室并不重要）：

* 如果你输入不同句子长度的纯文本数组，它将生成一个每个句子一行的矩阵，每个句子都由一个大小为 `MAX_SEQUENCE_LENGTH` 的数组表示。
* 该数组中的每个值使用 dictionary(`word_index`) 中对应的索引表示句子中的每个单词。 
* 比 `MAX_SEQUENCE_LENGTH` 短的序列用零填充以创建统一的长度。 

再次强调，这会在稍后的不计分实验中详细解释，所以现在不要太担心！

In [ ]:
tokenizer = Tokenizer(num_words=MAX_NB_WORDS)
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)

word_index = tokenizer.word_index
print('Found %s unique tokens.' % len(word_index))

data = pad_sequences(sequences, padding='post', maxlen=MAX_SEQUENCE_LENGTH)

print(data.shape)

print(data)

为了简化模型，你只需要获取正在检查的文本中出现的不同单词的嵌入。在本例中，你将仅过滤掉句子中出现的 11 个单词。第一个向量将是一个由零组成的数组，并将编码所有未知单词。

In [ ]:
embedding_matrix = np.zeros((len(word_index) + 1, EMBEDDING_DIM))
for word, i in word_index.items():
    embedding_vector = embeddings_index.get(word)
    if embedding_vector is not None:
        # 嵌入索引中未找到的单词将为全零。
        embedding_matrix[i] = embedding_vector
print(embedding_matrix.shape)

使用从预训练手套嵌入中提取的权重创建嵌入层。

In [ ]:
embedding_layer = Embedding(len(word_index) + 1,
                            EMBEDDING_DIM,
                            embeddings_initializer=tf.keras.initializers.Constant(embedding_matrix),
                            trainable=False)

使用前一层将输入分词数据转换为嵌入。检查嵌入的形状以确保该矩阵的最后一个维度包含句子中单词的嵌入。 

In [ ]:
embedding = embedding_layer(data)
print(embedding.shape)

<a name='2-2'></a>
### 2.2 - 笛卡尔平面上的可视化

现在，你将创建一个函数，它允许你在笛卡尔平面中可视化单词的编码。你将使用 PCA 将手套嵌入的 100 个特征减少到仅 2 个分量。 

In [ ]:
from sklearn.decomposition import PCA

def plot_words(embedding, sequences, sentence):
    pca = PCA(n_components=2)
    X_pca_train = pca.fit_transform(embedding[sentence,0:len(sequences[sentence]),:])


    fig, ax = plt.subplots(figsize=(12, 6)) 
    plt.rcParams['font.size'] = '12'
    ax.scatter(X_pca_train[:, 0], X_pca_train[:, 1])
    words = list(word_index.keys())
    for i, index in enumerate(sequences[sentence]):
        ax.annotate(words[index-1], (X_pca_train[i, 0], X_pca_train[i, 1]))


不错！现在你可以绘制每个句子的嵌入。每个图应该显示不同单词的嵌入。 


In [ ]:
plot_words(embedding, sequences, 0)

绘制第二句话的嵌入单词。回想一下，第二个句子包含与第一个句子相同的单词，只是顺序不同。你可以看到单词的顺序不会影响向量表示。 

In [ ]:
plot_words(embedding, sequences, 1)

<a name='3'></a>
## 3 - 语义和位置嵌入

接下来，你将把原始手套嵌入与之前计算的位置编码结合起来。在本练习中，你将在语义嵌入和位置嵌入之间使用 1:1 的权重比。 

In [ ]:
embedding2 = embedding * 1.0 + pos_encoding[:,:,:] * 1.0

plot_words(embedding2, sequences, 0)
plot_words(embedding2, sequences, 1)

哇看看情节之间的巨大差异！与原来的情节相比，这两个情节都发生了巨大的变化。请注意，在第二张图片（对应于相似单词不在一起的句子）中，非常不同的单词（例如 `red` 和 `wolf`）显得更接近。

现在你可以尝试不同的相对权重，看看这对句子中单词的向量表示有何强烈影响。

In [ ]:
W1 = 1 # 改变我
W2 = 10 # 改变我
embedding2 = embedding * W1 + pos_encoding[:,:,:] * W2
plot_words(embedding2, sequences, 0)
plot_words(embedding2, sequences, 1)

# 供参考
#['king queen man woman dog wolf football basketball red green yellow',
# 'man queen yellow basketball green dog  woman football  king red wolf']

如果设置 `W1 = 1` 和 `W2 = 10`，你可以看到单词的排列如何根据单词在句子中的位置开始采取顺时针或逆时针顺序。在这些参数下，位置编码向量在嵌入中占主导地位。 

现在尝试将权重反转为 `W1 = 10` 和 `W2 = 1`。观察到在这些参数下，该图类似于原始的嵌入可视化，并且绘制的单词的位置之间只有一些变化。

在前面的Transformer 作业中，词嵌入乘以`sqrt(EMBEDDING_DIM)`。在这种情况下，使用 `W1 = sqrt(EMBEDDING_DIM) = 10` 和 `W2 = 1` 是等效的。 

### 恭喜！
 
你已经完成了这个Notebook，并且对Transformer网络的输入有了更好的认识！ 

到目前为止，你已经： 

* 创建位置编码矩阵以可视化向量的关系属性
* 在笛卡尔平面上绘制嵌入和位置编码，以观察它们如何相互影响 


<div style="color:blue;">

<b>你应该记住什么</b>：
- 位置编码可以表示为彼此的线性函数，这使得模型可以根据单词的相对位置进行学习。
- 位置编码可以影响词嵌入，但如果位置编码的相对权重较小，则总和将保留单词的语义。

</div>